# 第 1 章互动实验：像素与色彩

这不是作业答案，而是一张可操纵的实验台。目标是先**看到现象**，再回到 `.py` 作业独立实现。

建议顺序：

1. 每个代码单元先预测结果，再运行。
2. 修改参数，至少制造一次失败案例。
3. 把观察写入 `实验记录.md`。
4. 不要把这里使用 OpenCV 基线的代码复制到“从零实现”作业。

## 0. 环境与数据

本单元会自动寻找课程根目录。若找不到图片，会给出明确错误，而不是静默生成不同数据。

In [ ]:
%matplotlib inline
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)

candidates = [
    Path.cwd(),
    Path.cwd() / 'cv',
]
COURSE_ROOT = next((p for p in candidates if (p / 'images').exists()), Path.cwd())

GRAY_PATH = COURSE_ROOT / 'images' / 'test_underexposed.jpg'
COLOR_PATH = COURSE_ROOT / 'images' / 'test_color.jpg'
gray = cv2.imread(str(GRAY_PATH), cv2.IMREAD_GRAYSCALE)
color_bgr = cv2.imread(str(COLOR_PATH), cv2.IMREAD_COLOR)
if gray is None or color_bgr is None:
    print('未找到生成图片，改用内存合成样例。也可运行 scripts/generate_sample_images.py。')
    rng = np.random.default_rng(42)
    gray = np.zeros((400, 500), dtype=np.uint8)
    gray[:300, :300] = rng.integers(20, 60, (300, 300), dtype=np.uint8)
    gray[100:300, 200:500] = rng.integers(160, 220, (200, 300), dtype=np.uint8)
    gray[50:100, 350:450] = 240
    color_bgr = np.zeros((300, 500, 3), dtype=np.uint8)
    color_bgr[:, :166, 2] = 255
    color_bgr[:, 167:333, 1] = 255
    color_bgr[:, 334:, 0] = 255

print('OpenCV:', cv2.__version__)
print('灰度图:', gray.shape, gray.dtype, (gray.min(), gray.max()))
print('彩色图:', color_bgr.shape, color_bgr.dtype)

## 1. 图像是带 dtype 的张量

先预测下面两个结果：

- `uint8(200) + uint8(100)` 是 300、255，还是 44？
- 转成整数后再相加，会发生什么？

图像处理中的许多“诡异黑点”其实不是算法问题，而是数据类型溢出。

In [ ]:
a = np.array([200], dtype=np.uint8)
b = np.array([100], dtype=np.uint8)

print('uint8 直接相加:', a + b)
print('转成 int16 相加:', a.astype(np.int16) + b.astype(np.int16))
print('饱和到 0-255:', np.clip(a.astype(np.int16) + b.astype(np.int16), 0, 255).astype(np.uint8))

### 小实验：亮度增益与截断

拖动增益。观察：

- 均值如何变化？
- 饱和到 255 的像素比例何时快速上升？
- 亮度增强为什么不能恢复已经饱和的细节？

In [ ]:
def show_gain(gain=1.0):
    enhanced = np.clip(gray.astype(np.float32) * gain, 0, 255).astype(np.uint8)
    saturated = np.mean(enhanced == 255)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(enhanced, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'gain={gain:.2f}, 饱和比例={saturated:.2%}')
    axes[0].axis('off')
    axes[1].hist(enhanced.ravel(), bins=256, range=(0, 256), color='black')
    axes[1].set_xlim(0, 255)
    axes[1].set_title('灰度直方图')
    plt.show()

interact(show_gain, gain=FloatSlider(min=0.3, max=3.0, step=0.1, value=1.0));

## 2. BGR、RGB 与 HSV

OpenCV 读入 BGR，Matplotlib 按 RGB 显示。如果直接 `plt.imshow(color_bgr)`，红蓝会交换。

下面同时显示：错误显示、正确显示、三个单通道。注意“某个区域在某通道更亮”意味着什么。

In [ ]:
color_rgb = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2RGB)
b, g, r = cv2.split(color_bgr)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
items = [
    (color_bgr, None, '错误：BGR 当 RGB'),
    (color_rgb, None, '正确：转成 RGB'),
    (b, 'gray', 'B 通道'),
    (g, 'gray', 'G 通道'),
    (r, 'gray', 'R 通道'),
]
for ax, (im, cmap, title) in zip(axes, items):
    ax.imshow(im, cmap=cmap, vmin=0 if cmap else None, vmax=255 if cmap else None)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

### 小实验：只修改 HSV 的 V 通道

拖动 `v_scale`。这是一种方便的亮度操作，但不是物理曝光模型：截断、色彩空间量化和相机响应曲线都会带来差异。

In [ ]:
def show_hsv_value_scale(v_scale=1.0):
    hsv = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * v_scale, 0, 255)
    result_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(color_rgb)
    axes[0].set_title('原图')
    axes[1].imshow(result_rgb)
    axes[1].set_title(f'V × {v_scale:.2f}')
    for ax in axes: ax.axis('off')
    plt.show()

interact(show_hsv_value_scale, v_scale=FloatSlider(min=0.3, max=2.5, step=0.1, value=1.0));

## 3. 直方图与 CDF

这一步只用 OpenCV 的实现作为**基线观察**。你的作业仍需独立实现 CDF 查表。

比较原图和均衡化图：均衡化通常增大对比度，但不保证视觉上更自然，也可能放大平坦暗区里的噪声。

In [ ]:
equalized_cv = cv2.equalizeHist(gray)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('原图')
axes[0, 1].imshow(equalized_cv, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('OpenCV 均衡化基线')
axes[1, 0].hist(gray.ravel(), bins=256, range=(0, 256), color='black')
axes[1, 0].set_title('原图直方图')
axes[1, 1].hist(equalized_cv.ravel(), bins=256, range=(0, 256), color='black')
axes[1, 1].set_title('均衡化后直方图')
for ax in axes[0]: ax.axis('off')
plt.tight_layout()

hist = np.bincount(gray.ravel(), minlength=256)
cdf = hist.cumsum() / hist.sum()
print('CDF 单调不减:', bool(np.all(np.diff(cdf) >= 0)))
print('CDF 末值:', cdf[-1])

### 思考

1. 为什么均衡化后的直方图并不会每个 bin 一样高？
2. 如果图像只有 4 个离散灰度值，映射后最多能有多少个灰度值？
3. CDF 中的零计数区间为什么需要特殊处理？

把答案写入实验记录，再去完成 `1-1_直方图均衡化.py`。

## 4. 噪声与滤波的匹配

下面控制椒盐噪声比例和中值窗口大小。请重点观察两件事：

- 噪声比例过高时，局部窗口的中位数也可能被污染。
- 窗口越大不一定越好：细线、小角点和纹理会被消除。

In [ ]:
def salt_pepper_for_demo(img, amount, seed=42):
    rng = np.random.default_rng(seed)
    noisy = img.copy()
    random_map = rng.random(img.shape)
    noisy[random_map < amount / 2] = 0
    noisy[(random_map >= amount / 2) & (random_map < amount)] = 255
    return noisy


def show_noise_filter(amount=0.05, kernel=3):
    if kernel % 2 == 0:
        kernel += 1
    noisy = salt_pepper_for_demo(gray, amount)
    median = cv2.medianBlur(noisy, kernel)
    mean = cv2.blur(noisy, (kernel, kernel))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, im, title in zip(
        axes,
        [gray, noisy, median, mean],
        ['原图', f'椒盐 {amount:.0%}', f'中值 {kernel}×{kernel}', f'均值 {kernel}×{kernel}'],
    ):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

interact(
    show_noise_filter,
    amount=FloatSlider(min=0.0, max=0.30, step=0.01, value=0.05),
    kernel=IntSlider(min=3, max=11, step=2, value=3),
);

## 5. 破坏实验

至少完成下面两个：

1. 把噪声比例提高到 20%，比较 3×3 与 9×9 中值滤波。
2. 用带细文字的真实图片测试大窗口滤波，记录细节如何消失。
3. 对高斯噪声使用中值滤波，再与高斯滤波比较。
4. 对低对比度但噪声很强的图片做全局均衡化，观察噪声是否被放大。

不要只保存“最好看”的结果。失败案例更能暴露方法的假设。

## 6. 离开 Notebook 完成作业

现在关闭提示，独立完成：

- `1-1_直方图均衡化.py`
- `1-2_通道分离与伪彩色.py`
- `1-3_椒盐噪声与中值滤波.py`

完成后运行：

```bash
python tests/check_progress.py
```

最后填写 `实验记录.md`。自动测试通过只是底线；你还需要能够解释每个结果为什么发生。